# EDA: CTU-13

Mechanical descriptive EDA driven by `openbotrisk.eda`.

In [1]:
import sys, time, datetime as dt
from pathlib import Path
import pandas as pd
import polars as pl

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from openbotrisk.eda.loaders import load_ctu13_meta
from openbotrisk.eda.report import write_report

DATA_DIR = REPO_ROOT / 'data' / 'CTU-13-Dataset'
REPORT_PATH = REPO_ROOT / 'working' / 'eda' / 'ctu-13.md'

t0 = time.time()
meta = load_ctu13_meta(DATA_DIR)
elapsed = time.time() - t0
df = meta['frame']
per = meta['per_scenario']
print(f'loaded in {elapsed:.1f}s; total rows={df.height:,}, cols={df.width}')

loaded in 1.4s; total rows=19,976,700, cols=16


In [2]:
def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f'{b:.1f} {u}'
        b /= 1024
    return f'{b:.1f} TB'

today = dt.date.today().isoformat()
size_lines = '\n'.join(f'- `{n}` — {fmt_size(s)}' for n, s in sorted(meta['file_sizes'].items(), key=lambda kv: int(kv[0].split("/")[0])))
access = (
    'Source: CTU-13 dataset, Czech Technical University (Stratosphere Lab) — '
    '`https://mcfp.felk.cvut.cz/publicDatasets/CTU-13-Dataset/`.\n\n'
    f'Local path: `{meta["data_dir"]}`\n\n'
    'File format: per-scenario `.binetflow` CSV (Argus bidirectional NetFlow export).\n\n'
    f'Date inspected: {today}.\n\n'
    'Files on disk (13 scenarios):\n' + size_lines
)
print(access[:600])

Source: CTU-13 dataset, Czech Technical University (Stratosphere Lab) — `https://mcfp.felk.cvut.cz/publicDatasets/CTU-13-Dataset/`.

Local path: `data/CTU-13-Dataset`

File format: per-scenario `.binetflow` CSV (Argus bidirectional NetFlow export).

Date inspected: 2026-05-23.

Files on disk (13 scenarios):
- `1/capture20110810.binetflow` — 368.7 MB
- `2/capture20110811.binetflow` — 235.8 MB
- `3/capture20110812.binetflow` — 610.0 MB
- `4/capture20110815.binetflow` — 146.7 MB
- `5/capture20110815-2.binetflow` — 16.9 MB
- `6/capture20110816.binetflow`


In [3]:
lines = ['| scenario | file | rows | size |', '|---|---|---|---|']
for _, r in per.sort_values('scenario', key=lambda s: s.astype(int)).iterrows():
    lines.append(f'| {r["scenario"]} | `{r["file"]}` | {int(r["rows"]):,} | {fmt_size(int(r["size_bytes"]))} |')
tmin = df['StartTime'].min(); tmax = df['StartTime'].max()
structure = (
    f'- 13 scenarios; each a separate Argus bidirectional netflow capture (`.binetflow` CSV).\n'
    f'- Record granularity: one bidirectional flow per row.\n'
    f'- Total rows (all scenarios concatenated): {df.height:,}; columns: {df.width} (14 native + `scenario` tag added by loader).\n'
    f'- Time range across all scenarios (`StartTime` as string): {tmin} to {tmax}.\n'
    f'- No join keys between scenarios; each capture is independent. Within a scenario, flows can be linked by `SrcAddr` / `DstAddr` / port tuples.\n\n'
    'Per-scenario row counts:\n\n' + '\n'.join(lines)
)
print(structure[:600])

- 13 scenarios; each a separate Argus bidirectional netflow capture (`.binetflow` CSV).
- Record granularity: one bidirectional flow per row.
- Total rows (all scenarios concatenated): 19,976,700; columns: 16 (14 native + `scenario` tag added by loader).
- Time range across all scenarios (`StartTime` as string): 2011/08/10 09:46:53.047277 to 2011/08/19 11:45:43.647861.
- No join keys between scenarios; each capture is independent. Within a scenario, flows can be linked by `SrcAddr` / `DstAddr` / port tuples.

Per-scenario row counts:

| scenario | file | rows | size |
|---|---|---|---|
| 1 | `


In [4]:
descs = {
    'StartTime': 'Flow start timestamp (string, microsecond precision)',
    'Dur': 'Flow duration in seconds',
    'Proto': 'L4 protocol (tcp, udp, icmp, ...)',
    'SrcAddr': 'Source IP address',
    'Sport': 'Source port',
    'Dir': 'Flow direction marker',
    'DstAddr': 'Destination IP address',
    'Dport': 'Destination port',
    'State': 'Argus flow state (connection state code)',
    'sTos': 'Source ToS byte',
    'dTos': 'Destination ToS byte',
    'TotPkts': 'Total packets in flow',
    'TotBytes': 'Total bytes in flow',
    'SrcBytes': 'Bytes sent by source',
    'Label': 'Argus label string; `flow=Background-...`, `flow=From-Botnet-...`, `flow=To-Botnet-...`, `flow=LEGITIMATE`, etc.',
    'scenario': 'Scenario folder id (1..13) — added by loader, not native.',
}
samp = df.head(1).to_pandas().iloc[0].to_dict() if df.height else {}
rows = ['| column | dtype | example | description |', '|---|---|---|---|']
for c in df.columns:
    ex = str(samp.get(c, ''))[:30]
    rows.append(f'| `{c}` | {df.schema[c]} | `{ex}` | {descs.get(c, "")} |')
schema_md = '\n'.join(rows) + '\n\nAll columns are real (non-anonymised) network metadata; IPs are real but from a controlled lab capture.'
print(schema_md[:600])

| column | dtype | example | description |
|---|---|---|---|
| `StartTime` | String | `2011/08/10 09:46:59.607825` | Flow start timestamp (string, microsecond precision) |
| `Dur` | Float64 | `1.026539` | Flow duration in seconds |
| `Proto` | String | `tcp` | L4 protocol (tcp, udp, icmp, ...) |
| `SrcAddr` | String | `94.44.127.113` | Source IP address |
| `Sport` | String | `1577` | Source port |
| `Dir` | String | `   ->` | Flow direction marker |
| `DstAddr` | String | `147.32.84.59` | Destination IP address |
| `Dport` | String | `6881` | Destination port |
| `State` | String | `S_RA` | A


In [5]:
# Derive a coarse label class from the Label string.
lbl_classes = (
    df.with_columns(
        pl.when(pl.col('Label').str.contains('(?i)botnet')).then(pl.lit('Botnet'))
        .when(pl.col('Label').str.contains('(?i)legitimate')).then(pl.lit('Legitimate'))
        .when(pl.col('Label').str.contains('(?i)background')).then(pl.lit('Background'))
        .otherwise(pl.lit('Other'))
        .alias('label_class')
    )
    .group_by('label_class').len().sort('len', descending=True)
)
lc_df = lbl_classes.to_pandas()
total = int(lc_df['len'].sum())
lines = ['| label_class | count | rate |', '|---|---|---|']
for _, r in lc_df.iterrows():
    lines.append(f'| {r["label_class"]} | {int(r["len"]):,} | {r["len"]/total:.5f} |')
n_distinct = df.select(pl.col('Label').n_unique()).item()
label_md = (
    'Label column: `Label` (free-text Argus annotation). Coarsened to three classes by '
    'string match: `Botnet` (contains "Botnet"), `Legitimate`, `Background`, else `Other`.\n\n'
    + '\n'.join(lines) + f'\n\nDistinct raw label strings across all scenarios: {n_distinct}. '
    'Botnet traffic is heavily outnumbered by background. Background flows are unlabelled real traffic and '
    'should not be treated as confirmed-legitimate.'
)
print(label_md)

Label column: `Label` (free-text Argus annotation). Coarsened to three classes by string match: `Botnet` (contains "Botnet"), `Legitimate`, `Background`, else `Other`.

| label_class | count | rate |
|---|---|---|
| Background | 19,175,568 | 0.95990 |
| Botnet | 444,699 | 0.02226 |
| Other | 356,433 | 0.01784 |

Distinct raw label strings across all scenarios: 1400. Botnet traffic is heavily outnumbered by background. Background flows are unlabelled real traffic and should not be treated as confirmed-legitimate.


In [6]:
ident_cols = ['SrcAddr', 'DstAddr', 'Sport', 'Dport', 'Proto']
lines = ['| column | n_unique | null_rate | role |', '|---|---|---|---|']
n = df.height
roles = {
    'SrcAddr': 'source IP — strongest actor identifier (per-host)',
    'DstAddr': 'destination IP — actor identifier for the contacted endpoint',
    'Sport': 'source port (often ephemeral)',
    'Dport': 'destination port (service identifier)',
    'Proto': 'L4 protocol',
}
for c in ident_cols:
    uniq = df.select(pl.col(c).n_unique()).item()
    nulls = df.select(pl.col(c).is_null().sum()).item()
    lines.append(f'| `{c}` | {uniq:,} | {nulls/n:.4f} | {roles[c]} |')
ident_md = (
    'Identifiers are real IP addresses and ports (no user IDs, no device fingerprints).\n\n'
    + '\n'.join(lines) + '\n\n'
    'Within a scenario, the infected hosts have known IPs (documented in each scenario `README`). '
    'There is no user-level identifier; the actor unit is the host (IP) inside the captured network.'
)
print(ident_md)

Identifiers are real IP addresses and ports (no user IDs, no device fingerprints).

| column | n_unique | null_rate | role |
|---|---|---|---|
| `SrcAddr` | 2,031,491 | 0.0000 | source IP — strongest actor identifier (per-host) |
| `DstAddr` | 531,964 | 0.0000 | destination IP — actor identifier for the contacted endpoint |
| `Sport` | 118,218 | 0.0102 | source port (often ephemeral) |
| `Dport` | 108,451 | 0.0097 | destination port (service identifier) |
| `Proto` | 19 | 0.0000 | L4 protocol |

Within a scenario, the infected hosts have known IPs (documented in each scenario `README`). There is no user-level identifier; the actor unit is the host (IP) inside the captured network.


In [7]:
# Per-scenario time spans (StartTime sorted lexicographically equals chronological for the fixed format).
per_time = (
    df.group_by('scenario').agg([
        pl.col('StartTime').min().alias('t_min'),
        pl.col('StartTime').max().alias('t_max'),
        pl.len().alias('rows'),
    ]).sort('scenario')
).to_pandas()
lines = ['| scenario | t_min | t_max | rows |', '|---|---|---|---|']
for _, r in per_time.iterrows():
    lines.append(f'| {r["scenario"]} | {r["t_min"]} | {r["t_max"]} | {int(r["rows"]):,} |')
temporal_md = (
    '- `StartTime`: string timestamp with microsecond precision (`YYYY/MM/DD HH:MM:SS.ffffff`).\n'
    '- `Dur`: float seconds duration of the flow.\n'
    '- Granularity: microsecond start; flow durations span sub-second to hours.\n'
    '- Each scenario is a contiguous capture window of hours; scenarios are not contemporaneous.\n\n'
    'Per-scenario time spans:\n\n' + '\n'.join(lines)
)
print(temporal_md[:600])

- `StartTime`: string timestamp with microsecond precision (`YYYY/MM/DD HH:MM:SS.ffffff`).
- `Dur`: float seconds duration of the flow.
- Granularity: microsecond start; flow durations span sub-second to hours.
- Each scenario is a contiguous capture window of hours; scenarios are not contemporaneous.

Per-scenario time spans:

| scenario | t_min | t_max | rows |
|---|---|---|---|
| 1 | 2011/08/10 09:46:53.047277 | 2011/08/10 15:54:07.368340 | 2,824,636 |
| 10 | 2011/08/18 09:56:29.146156 | 2011/08/18 15:04:59.744388 | 1,309,791 |
| 11 | 2011/08/18 15:39:35.087798 | 2011/08/18 15:55:46.379941 


In [8]:
null_counts = {c: df.select(pl.col(c).is_null().sum()).item() for c in df.columns}
lines = ['| column | null_count | null_rate |', '|---|---|---|']
for c, k in sorted(null_counts.items(), key=lambda kv: -kv[1]):
    lines.append(f'| `{c}` | {k:,} | {k/n:.5f} |')
hi = [c for c, k in null_counts.items() if k / n > 0.2]
missing_md = (
    f'Overall null cells: {sum(null_counts.values()):,} / {n * df.width:,} '
    f'({sum(null_counts.values()) / (n * df.width):.2%}).\n\n'
    'Columns with >20% missingness: ' + (', '.join(f'`{c}`' for c in hi) if hi else '_none_') + '.\n\n'
    'Per-column nulls (all scenarios concatenated):\n\n' + '\n'.join(lines) + '\n\n'
    '`sTos` / `dTos` are commonly null because routers do not always set ToS. '
    'Port nulls occur for connectionless / ICMP / ARP flows.'
)
print(missing_md[:600])

Overall null cells: 2,337,061 / 319,627,200 (0.73%).

Columns with >20% missingness: _none_.

Per-column nulls (all scenarios concatenated):

| column | null_count | null_rate |
|---|---|---|
| `dTos` | 1,718,011 | 0.08600 |
| `sTos` | 220,525 | 0.01104 |
| `Sport` | 203,085 | 0.01017 |
| `Dport` | 194,062 | 0.00971 |
| `State` | 1,378 | 0.00007 |
| `StartTime` | 0 | 0.00000 |
| `Dur` | 0 | 0.00000 |
| `Proto` | 0 | 0.00000 |
| `SrcAddr` | 0 | 0.00000 |
| `Dir` | 0 | 0.00000 |
| `DstAddr` | 0 | 0.00000 |
| `TotPkts` | 0 | 0.00000 |
| `TotBytes` | 0 | 0.00000 |
| `SrcBytes` | 0 | 0.00000 |
| `L


In [9]:
quirks_md = (
    '- 13 separate captures; each scenario uses a different botnet family (per scenario `README`). They are not a single time series.\n'
    '- `Label` is a free-text Argus annotation, not a clean class column; coarsening requires substring matching.\n'
    '- `Background` flows are unlabelled real traffic from the same network — they are NOT verified-legitimate.\n'
    '- `Sport` / `Dport` are stored as strings (some non-numeric values like `0x*` exist), not integers.\n'
    '- IPs are real and from a single university network capture; the actor unit is a host, not a user.\n'
    '- Concatenating all scenarios into one frame is fine for descriptive stats but mixes captures from different dates and botnet families.'
)
print(quirks_md)

- 13 separate captures; each scenario uses a different botnet family (per scenario `README`). They are not a single time series.
- `Label` is a free-text Argus annotation, not a clean class column; coarsening requires substring matching.
- `Background` flows are unlabelled real traffic from the same network — they are NOT verified-legitimate.
- `Sport` / `Dport` are stored as strings (some non-numeric values like `0x*` exist), not integers.
- IPs are real and from a single university network capture; the actor unit is a host, not a user.
- Concatenating all scenarios into one frame is fine for descriptive stats but mixes captures from different dates and botnet families.


In [10]:
repro_md = (
    'Generated by `notebooks/eda/ctu-13.ipynb` which calls '
    '`openbotrisk.eda.loaders.load_ctu13_meta` (polars read of each `*.binetflow`, then vertical concat).\n\n'
    'Run with:\n\n'
    '```bash\n'
    'jupyter nbconvert --to notebook --execute --inplace \\\n'
    '  notebooks/eda/ctu-13.ipynb \\\n'
    '  --ExecutePreprocessor.timeout=600\n'
    '```\n\n'
    f'Loader runtime on this machine: {elapsed:.1f}s for all 13 scenarios concatenated in-memory with polars.'
)
print(repro_md)

Generated by `notebooks/eda/ctu-13.ipynb` which calls `openbotrisk.eda.loaders.load_ctu13_meta` (polars read of each `*.binetflow`, then vertical concat).

Run with:

```bash
jupyter nbconvert --to notebook --execute --inplace \
  notebooks/eda/ctu-13.ipynb \
  --ExecutePreprocessor.timeout=600
```

Loader runtime on this machine: 1.4s for all 13 scenarios concatenated in-memory with polars.


In [11]:
stats = {
    'Access': access,
    'Structure': structure,
    'Schema': schema_md,
    'Label': label_md,
    'Identifier inventory': ident_md,
    'Temporal structure': temporal_md,
    'Missing data': missing_md,
    'Quirks and observations': quirks_md,
    'Reproduction': repro_md,
}
out = write_report(stats, 'CTU-13', REPORT_PATH)
print('wrote', out)

wrote working/eda/ctu-13.md
